In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F 
device='cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using {device} device")
block_size=8
batch_size=4

Using cpu device


In [3]:
from google.colab import drive
drive.mount('/content/drive')


ModuleNotFoundError: No module named 'google.colab'

In [31]:
with open ('/content/drive/MyDrive/pg22566.txt', 'r', encoding="utf-8") as f:
    text = f.read()
print(len(text))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/pg22566.txt'

In [2]:
from pathlib import Path
text = Path("pg22566.txt").read_text(encoding="utf-8")
print(len(text))

252022


In [3]:
print(text[:200])

The Project Gutenberg eBook of Dorothy and the Wizard in Oz
    
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no rest


In [4]:
chars=sorted(set(text))
print(chars)

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™', '\ufeff']


In [6]:
print(len(chars))

92


In [7]:
string_to_int = {ch:i for i, ch in enumerate(chars)}
int_to_string = {i:ch for i, ch in enumerate(chars)}
encode=lambda s: [string_to_int[c] for c in s]
decode=lambda l: ''.join([int_to_string[i] for i in l])
print(encode("hello world"))
print(decode(encode("hello world")))


[65, 62, 69, 69, 72, 1, 80, 72, 75, 69, 61]
hello world


In [8]:
data=torch.tensor(encode(text),dtype=torch.long)
print(data[:100])

tensor([91, 48, 65, 62,  1, 44, 75, 72, 67, 62, 60, 77,  1, 35, 78, 77, 62, 71,
        59, 62, 75, 64,  1, 62, 30, 72, 72, 68,  1, 72, 63,  1, 32, 72, 75, 72,
        77, 65, 82,  1, 58, 71, 61,  1, 77, 65, 62,  1, 51, 66, 83, 58, 75, 61,
         1, 66, 71,  1, 43, 83,  0,  1,  1,  1,  1,  0, 48, 65, 66, 76,  1, 62,
        59, 72, 72, 68,  1, 66, 76,  1, 63, 72, 75,  1, 77, 65, 62,  1, 78, 76,
        62,  1, 72, 63,  1, 58, 71, 82, 72, 71])


In [9]:
n=int(0.8*len(data))
train_data=data[:n]
val_data=data[n:]
def get_batch(split):
    data=train_data if split=='train' else val_data
    ix=torch.randint(len(data)-block_size,(batch_size,))
    x=torch.stack([data[i:i+block_size] for i in ix])
    y=torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x.to(device),y.to(device)
x,y=get_batch('train')
print('inputs:')
print(x.shape)
print(x)
print('targets:')
print(y.shape)
print(y)

inputs:
torch.Size([4, 8])
tensor([[70, 66, 76, 77, 58, 68, 62, 76],
        [63,  1, 70, 82,  1, 68, 66, 71],
        [ 0,  3, 36, 62, 75, 62, 13, 13],
        [61, 62, 76, 62, 75, 79, 62,  1]])
targets:
torch.Size([4, 8])
tensor([[66, 76, 77, 58, 68, 62, 76,  1],
        [ 1, 70, 82,  1, 68, 66, 71, 64],
        [ 3, 36, 62, 75, 62, 13, 13, 76],
        [62, 76, 62, 75, 79, 62,  1, 77]])


In [10]:
block_size=8
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
    context=x[:t+1]
    target=y[:t+1]
    print(f"when input is {context} the target is {target}")

when input is tensor([91]) the target is tensor([48])
when input is tensor([91, 48]) the target is tensor([48, 65])
when input is tensor([91, 48, 65]) the target is tensor([48, 65, 62])
when input is tensor([91, 48, 65, 62]) the target is tensor([48, 65, 62,  1])
when input is tensor([91, 48, 65, 62,  1]) the target is tensor([48, 65, 62,  1, 44])
when input is tensor([91, 48, 65, 62,  1, 44]) the target is tensor([48, 65, 62,  1, 44, 75])
when input is tensor([91, 48, 65, 62,  1, 44, 75]) the target is tensor([48, 65, 62,  1, 44, 75, 72])
when input is tensor([91, 48, 65, 62,  1, 44, 75, 72]) the target is tensor([48, 65, 62,  1, 44, 75, 72, 67])


In [11]:
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
    context=x[:t+1]
    target=y[:t+1]
    print(f"when input is {context} the target is {target}")

when input is tensor([91]) the target is tensor([48])
when input is tensor([91, 48]) the target is tensor([48, 65])
when input is tensor([91, 48, 65]) the target is tensor([48, 65, 62])
when input is tensor([91, 48, 65, 62]) the target is tensor([48, 65, 62,  1])
when input is tensor([91, 48, 65, 62,  1]) the target is tensor([48, 65, 62,  1, 44])
when input is tensor([91, 48, 65, 62,  1, 44]) the target is tensor([48, 65, 62,  1, 44, 75])
when input is tensor([91, 48, 65, 62,  1, 44, 75]) the target is tensor([48, 65, 62,  1, 44, 75, 72])
when input is tensor([91, 48, 65, 62,  1, 44, 75, 72]) the target is tensor([48, 65, 62,  1, 44, 75, 72, 67])


In [12]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)
    def forward(self, idx, targets=None):
        logits=self.token_embedding_table(idx)
        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)
        return logits,loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self.forward(idx)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx
    
model=BigramLanguageModel(vocab_size=len(chars))
m=model.to(device)
context=torch.zeros((1,1),dtype=torch.long,device=device)
generated_chars=model.generate(context,max_new_tokens=500)[0].tolist()
print(generated_chars)

[0, 59, 14, 67, 68, 29, 29, 24, 90, 39, 91, 84, 40, 17, 9, 84, 52, 73, 27, 19, 36, 38, 29, 29, 36, 57, 19, 76, 16, 22, 45, 18, 25, 38, 69, 46, 76, 39, 7, 9, 36, 64, 60, 5, 57, 11, 44, 12, 76, 29, 51, 89, 4, 90, 64, 72, 40, 4, 45, 40, 50, 13, 50, 88, 61, 70, 13, 73, 87, 29, 21, 9, 30, 55, 77, 41, 19, 49, 72, 45, 85, 66, 19, 6, 21, 47, 44, 12, 68, 7, 52, 29, 21, 0, 1, 47, 86, 40, 1, 13, 24, 1, 49, 21, 77, 24, 76, 9, 33, 87, 12, 32, 32, 43, 83, 69, 8, 3, 24, 84, 65, 64, 84, 43, 25, 30, 79, 81, 60, 40, 43, 54, 91, 36, 66, 33, 74, 1, 22, 75, 33, 70, 71, 39, 87, 21, 14, 30, 75, 61, 28, 84, 56, 3, 31, 63, 41, 9, 69, 31, 38, 71, 80, 35, 78, 0, 3, 65, 40, 0, 83, 52, 3, 32, 80, 5, 91, 50, 19, 48, 26, 6, 90, 49, 10, 33, 46, 60, 3, 88, 19, 6, 7, 71, 81, 67, 49, 67, 47, 44, 85, 3, 52, 75, 30, 75, 3, 14, 67, 25, 55, 28, 24, 61, 17, 49, 24, 49, 54, 56, 66, 75, 57, 60, 55, 78, 4, 48, 50, 47, 58, 16, 62, 38, 44, 45, 0, 69, 46, 64, 62, 34, 20, 2, 61, 77, 24, 24, 87, 29, 62, 34, 69, 16, 62, 36, 84, 88, 4

In [13]:
optim=torch.optim.AdamW(model.parameters(),lr=3e-4)
for i in range(40000):
    xb,yb=get_batch('train')
    logits,loss=model.forward(xb,yb)
    optim.zero_grad(set_to_none=True)
    loss.backward()
    optim.step()
    optim.zero_grad(set_to_none=True)
print(loss.item())

2.177081346511841


In [14]:
@torch.no_grad()
def estimate_loss(eval_iters=200):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

losses = estimate_loss()
print(f"train loss: {losses['train']:.4f}")
print(f"val loss: {losses['val']:.4f}")


train loss: 2.4819
val loss: 2.6327


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

@torch.no_grad()
def plot_val_roc_auc(eval_batches=30):
    model.eval()
    y_true_all = []
    y_score_all = []
    vocab_size = len(chars)

    for _ in range(eval_batches):
        X, Y = get_batch("val")
        logits, _ = model(X)
        probs = F.softmax(logits, dim=-1)

        y_true_all.append(Y.reshape(-1).cpu().numpy())
        y_score_all.append(probs.reshape(-1, vocab_size).cpu().numpy())

    model.train()

    y_true = np.concatenate(y_true_all)
    y_score = np.concatenate(y_score_all)

    y_true_bin = label_binarize(y_true, classes=np.arange(vocab_size))
    fpr, tpr, _ = roc_curve(y_true_bin.ravel(), y_score.ravel())
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"micro-average ROC (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Validation ROC Curve")
    plt.legend(loc="lower right")
    plt.grid(alpha=0.2)
    plt.show()

    print(f"Validation micro-average ROC-AUC: {roc_auc:.4f}")

plot_val_roc_auc(eval_batches=30)


In [47]:
context=torch.zeros((1,1),dtype=torch.long,device=device)
generated_chars=model.generate(context,max_new_tokens=500)[0].tolist()
print(decode(generated_chars))


[VOz, 2—
E.
teas, ley Alo rallelixf o popo ththod. answe mer. sif m; bou.

ay o ake. my sho taroousitere't der wed be o ay o, h ino " OUinomy hin Z#Um binoske te thethe tid
lin, wathe aw sti jan?L?0yond s te ado tay!”﻿4uny hey, t Gzabed-boreths *Carandrey wherd stad alith!'s herr hyora iste we Thed w? toicticofaglwer wate toupu d t, ar awe

ste
" to hiedicchtt he't tteewarard

athen ffus, ming tht ad t on', waice?]I"
L.gsmyon uGangrgor?"Song, t theloumesherelo cotcked coit Pr wald sthad izandoor


In [48]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)
    def forward(self, idx, targets=None):
        logits=self.token_embedding_table(idx)
        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)
        return logits,loss
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits,self.forward(idx)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            idx_next=torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx
    
model=BigramLanguageModel(vocab_size=len(chars))
m=model.to(device)
context=torch.zeros((1,1),dtype=torch.long,device=device)
generated_chars=model.generate(context,max_new_tokens=500)[0].tolist()
print(generated_chars)

UnboundLocalError: cannot access local variable 'logits' where it is not associated with a value